In [13]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [14]:
dataset_path = '../data/personality_dataset.csv'
df = pd.read_csv(dataset_path)
print(f"Dataset loaded. Shape: {df.shape}")

Dataset loaded. Shape: (2900, 8)


In [15]:
target_col = 'Personality'
X = df.drop(columns=[target_col])
y = df[target_col]

# Map targets to binary (0: Introvert, 1: Extrovert)
y_mapped = y.map({'Introvert': 0, 'Extrovert': 1})

# Define feature types
num_features = ['Time_spent_Alone', 'Social_event_attendance', 'Going_outside', 'Friends_circle_size', 'Post_frequency']
cat_features = ['Stage_fear', 'Drained_after_socializing']

# Build the transformers
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

In [16]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_mapped, test_size=0.2, random_state=42, stratify=y_mapped
)

# Train
model_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = model_pipeline.predict(X_test)
print(f"Validation Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Introvert', 'Extrovert']))

Validation Accuracy: 0.9103

Classification Report:
               precision    recall  f1-score   support

   Introvert       0.90      0.91      0.91       282
   Extrovert       0.92      0.91      0.91       298

    accuracy                           0.91       580
   macro avg       0.91      0.91      0.91       580
weighted avg       0.91      0.91      0.91       580



In [18]:
try:
    joblib.dump(model_pipeline, '../model/model_pipeline.pkl')
    print("Model pipeline successfully saved to the root directory as 'model_pipeline.pkl'!")
except NameError:
    print("Error: 'model_pipeline' is not defined. Please run your training cells first, then run this cell.")

Model pipeline successfully saved to the root directory as 'model_pipeline.pkl'!
